In [ ]:
from dotenv import load_dotenv
load_dotenv()

import re
import math
import voyageai

client = voyageai.Client()

In [ ]:
def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    inp = chunks if is_list else [chunks]
    result = client.embed(inp, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# --- Vector Database ---
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.documents = []
        self._vector_dim = None

    def add_vector(self, vector, document):
        if not self.vectors: self._vector_dim = len(vector)
        self.vectors.append(list(vector))
        self.documents.append(document)

    def search(self, query_vector, k=1):
        distances = []
        for i, stored in enumerate(self.vectors):
            dot = sum(a * b for a, b in zip(query_vector, stored))
            mag1 = math.sqrt(sum(x*x for x in query_vector))
            mag2 = math.sqrt(sum(x*x for x in stored))
            sim = dot / (mag1 * mag2) if mag1 and mag2 else 0
            distances.append((1.0 - sim, self.documents[i]))
        distances.sort(key=lambda x: x[0])
        return [(doc, dist) for dist, doc in distances[:k]]

    def __repr__(self):
        return f"VectorIndex(count={len(self.vectors)}, dim={self._vector_dim})"

In [ ]:
# Step 1: Chunk
with open("report.md", "r") as f:
    text = f.read()
chunks = chunk_by_section(text)
print(f"{len(chunks)} chunks")

In [ ]:
# Step 2: Embed all chunks
embeddings = generate_embedding(chunks, input_type="document")
print(f"{len(embeddings)} embeddings, {len(embeddings[0])} dims")

In [ ]:
# Step 3: Store in vector index
store = VectorIndex()
for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})
print(store)

In [ ]:
# Step 4+5: Query and search
query = "What did the software engineering department do last year?"
query_emb = generate_embedding(query, input_type="query")
results = store.search(query_emb, k=2)

for doc, dist in results:
    first_line = doc["content"].strip().split("\n")[0][:80]
    print(f"  distance={dist:.4f}: {first_line}")